# Embedding Extraction Pipeline

Extract text and image embeddings using pretrained encoders from `src/` modules.
- Text: PhoBERT (768-dim)
- Image: Vision Transformer (768-dim)
- Saves embeddings for train/val/test splits

## 1. Setup & Imports

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

# Add project root to path
project_root = Path(r"C:\Users\Vivobook\Documents\mm_detect")
sys.path.insert(0, str(project_root))

# Import from src
from src.config.env_config import config
from src.utils.seed import set_seed
from src.utils.logger import setup_logger
from src.models.encoders.text_encoder import TextEncoder
from src.models.encoders.image_encoder import ImageEncoder

# Setup logging and seed
logger = setup_logger(__name__)
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")

C:\Users\Vivobook\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Device: cpu


## 2. Load Configuration

In [3]:
# Display configuration settings
print("\n" + "="*80)
print("CONFIGURATION SETTINGS")
print("="*80)
print(f"Data paths:")
print(f"  Raw: {config.data_raw_dir}")
print(f"  Processed: {config.data_processed_dir}")
print(f"  Embeddings: {config.embeddings_dir}")
print(f"\nModel settings:")
print(f"  Text model: {config.text_model_name}")
print(f"  Image model: {config.image_model_name}")
print(f"  Embedding dim: {config.embedding_dim}")
print(f"  Text max length: {config.text_max_length}")
print(f"  Batch size: {config.batch_size}")


CONFIGURATION SETTINGS
Data paths:
  Raw: c:\Users\Vivobook\Documents\mm_detect\data\raw
  Processed: c:\Users\Vivobook\Documents\mm_detect\data\processed
  Embeddings: c:\Users\Vivobook\Documents\mm_detect\data\embeddings

Model settings:
  Text model: vinai/phobert-base
  Image model: google/vit-base-patch16-224
  Embedding dim: 768
  Text max length: 256
  Batch size: 32


## 3. Load Dataset

In [4]:
# Load train/val/test splits created by preprocessing
processed_dir = Path(config.data_processed_dir)

train_df = pd.read_csv(processed_dir / 'train.csv')
val_df = pd.read_csv(processed_dir / 'val.csv')
test_df = pd.read_csv(processed_dir / 'test.csv')

print(f"\n" + "="*80)
print("DATASET LOADED")
print("="*80)
print(f"Train set: {len(train_df)} samples")
print(f"Val set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)} samples")
print(f"\nColumns: {list(train_df.columns)[:5]}...")


DATASET LOADED
Train set: 11674 samples
Val set: 2502 samples
Test set: 2502 samples
Total: 16678 samples

Columns: ['id', 'page_id', 'page_name', 'ad_creation_time', 'ad_delivery_start_time']...


## 4. Initialize Encoders

In [5]:
# Initialize encoders (PyTorch modules)
print("\nInitializing encoders...")
text_encoder = TextEncoder(
    model_name=config.text_model_name,
    max_length=config.text_max_length
).to(device)

image_encoder = ImageEncoder(
    model_name=config.image_model_name
).to(device)

print(f"✓ Text encoder: {config.text_model_name}")
print(f"✓ Image encoder: {config.image_model_name}")


Initializing encoders...


C:\Users\Vivobook\AppData\Roaming\Python\Python313\site-packages\transformers\models\vit\feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Text encoder: vinai/phobert-base
✓ Image encoder: google/vit-base-patch16-224


## 5. Extract Embeddings

In [6]:
def extract_embeddings_split(df, split_name='train'):
    """Extract text and image embeddings for a dataset split"""
    
    # Identify text columns
    text_cols = [col for col in df.columns 
                 if 'body' in col.lower() or 'title' in col.lower() or 'creative' in col.lower()]
    if not text_cols:
        text_cols = [df.columns[0]]
    
    print(f"\nUsing text columns: {text_cols}")
    
    # Prepare texts
    texts = []
    for idx, row in df.iterrows():
        parts = []
        for col in text_cols:
            if col in df.columns and pd.notna(row[col]):
                parts.append(str(row[col]))
        text = " ".join(parts) if parts else ""
        texts.append(text)
    
    # Extract text embeddings with batching
    print(f"\nExtracting text embeddings for {split_name} set...")
    text_embeddings = []
    batch_size = config.batch_size
    
    for i in tqdm(range(0, len(texts), batch_size), desc=f"{split_name} text"):
        batch = texts[i:i+batch_size]
        # Ensure all texts are strings
        batch = [str(t) if t else "" for t in batch]
        
        with torch.no_grad():
            embeddings = text_encoder(batch, device)
        text_embeddings.append(embeddings.cpu().numpy())
    
    text_embeddings = np.vstack(text_embeddings)
    print(f"✓ Text embeddings shape: {text_embeddings.shape}")
    
    # Extract image embeddings
    print(f"\nExtracting image embeddings for {split_name} set...")
    image_dir = Path(config.data_raw_dir) / 'ad_images'
    image_embeddings = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{split_name} image"):
        # Find image with various extensions
        image_id = str(row.get('image_id', row.get('id', idx)))
        image_path = None
        
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            candidate = image_dir / f"{image_id}{ext}"
            if candidate.exists():
                image_path = candidate
                break
        
        if image_path:
            try:
                img = Image.open(image_path).convert('RGB')
                with torch.no_grad():
                    embedding = image_encoder([img], device)
                image_embeddings.append(embedding.cpu().numpy())
            except Exception as e:
                # Use zero embedding if image cannot be loaded
                image_embeddings.append(np.zeros((1, config.embedding_dim), dtype=np.float32))
        else:
            # Use zero embedding if image not found
            image_embeddings.append(np.zeros((1, config.embedding_dim), dtype=np.float32))
    
    image_embeddings = np.vstack(image_embeddings)
    print(f"✓ Image embeddings shape: {image_embeddings.shape}")
    
    return text_embeddings, image_embeddings

# Extract embeddings for all splits
print("\n" + "="*80)
print("EMBEDDING EXTRACTION")
print("="*80)

train_text, train_image = extract_embeddings_split(train_df, 'train')
val_text, val_image = extract_embeddings_split(val_df, 'val')
test_text, test_image = extract_embeddings_split(test_df, 'test')


EMBEDDING EXTRACTION

Using text columns: ['ad_creative_bodies', 'ad_creative_link_titles', 'ad_creative_bodies_length', 'ad_creative_link_titles_length']

Extracting text embeddings for train set...


train text: 100%|██████████| 365/365 [51:54<00:00,  8.53s/it] 


✓ Text embeddings shape: (11674, 768)

Extracting image embeddings for train set...


train image: 100%|██████████| 11674/11674 [54:55<00:00,  3.54it/s] 


✓ Image embeddings shape: (11674, 768)

Using text columns: ['ad_creative_bodies', 'ad_creative_link_titles', 'ad_creative_bodies_length', 'ad_creative_link_titles_length']

Extracting text embeddings for val set...


...

✓ Image embeddings shape: (2502, 768)

Using text columns: ['ad_creative_bodies', 'ad_creative_link_titles', 'ad_creative_bodies_length', 'ad_creative_link_titles_length']

Extracting text embeddings for test set...


test text: 100%|██████████| 79/79 [10:23<00:00,  7.89s/it]


✓ Text embeddings shape: (2502, 768)

Extracting image embeddings for test set...


test image: 100%|██████████| 2502/2502 [11:21<00:00,  3.67it/s]


✓ Image embeddings shape: (2502, 768)


## 6. Save Embeddings

In [7]:
# Create embeddings directory
embeddings_dir = Path(config.embeddings_dir)
embeddings_dir.mkdir(parents=True, exist_ok=True)

print(f"\n" + "="*80)
print("SAVING EMBEDDINGS")
print("="*80)
print(f"Output directory: {embeddings_dir}")

# Save text embeddings
np.save(embeddings_dir / 'train_text_embeddings.npy', train_text)
np.save(embeddings_dir / 'val_text_embeddings.npy', val_text)
np.save(embeddings_dir / 'test_text_embeddings.npy', test_text)
print(f"✓ Text embeddings saved")

# Save image embeddings
np.save(embeddings_dir / 'train_image_embeddings.npy', train_image)
np.save(embeddings_dir / 'val_image_embeddings.npy', val_image)
np.save(embeddings_dir / 'test_image_embeddings.npy', test_image)
print(f"✓ Image embeddings saved")

# Save labels
label_col = 'misinformation' if 'misinformation' in train_df.columns else 'label'
np.save(embeddings_dir / 'train_labels.npy', train_df[label_col].values)
np.save(embeddings_dir / 'val_labels.npy', val_df[label_col].values)
np.save(embeddings_dir / 'test_labels.npy', test_df[label_col].values)
print(f"✓ Labels saved")

print(f"\n✓ All embeddings saved to {embeddings_dir}")


SAVING EMBEDDINGS
Output directory: c:\Users\Vivobook\Documents\mm_detect\data\embeddings
✓ Text embeddings saved
✓ Image embeddings saved
✓ Labels saved

✓ All embeddings saved to c:\Users\Vivobook\Documents\mm_detect\data\embeddings


## 7. Summary

In [8]:
import json

# Create summary
summary = {
    'embedding_config': {
        'text_model': config.text_model_name,
        'image_model': config.image_model_name,
        'embedding_dim': config.embedding_dim,
        'device': str(device)
    },
    'splits': {
        'train': {
            'text_shape': train_text.shape,
            'image_shape': train_image.shape,
            'labels_shape': train_df[label_col].shape,
            'label_dist': dict(train_df[label_col].value_counts().sort_index())
        },
        'val': {
            'text_shape': val_text.shape,
            'image_shape': val_image.shape,
            'labels_shape': val_df[label_col].shape,
            'label_dist': dict(val_df[label_col].value_counts().sort_index())
        },
        'test': {
            'text_shape': test_text.shape,
            'image_shape': test_image.shape,
            'labels_shape': test_df[label_col].shape,
            'label_dist': dict(test_df[label_col].value_counts().sort_index())
        }
    }
}

# Save summary
with open(embeddings_dir / 'embedding_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print("\n" + "="*80)
print("EMBEDDING EXTRACTION SUMMARY")
print("="*80)
print(f"\nTrain set:")
print(f"  Text shape: {summary['splits']['train']['text_shape']}")
print(f"  Image shape: {summary['splits']['train']['image_shape']}")
print(f"  Labels: {summary['splits']['train']['label_dist']}")

print(f"\nVal set:")
print(f"  Text shape: {summary['splits']['val']['text_shape']}")
print(f"  Image shape: {summary['splits']['val']['image_shape']}")
print(f"  Labels: {summary['splits']['val']['label_dist']}")

print(f"\nTest set:")
print(f"  Text shape: {summary['splits']['test']['text_shape']}")
print(f"  Image shape: {summary['splits']['test']['image_shape']}")
print(f"  Labels: {summary['splits']['test']['label_dist']}")

print(f"\n✓ Pipeline complete! Summary saved to {embeddings_dir / 'embedding_summary.json'}")


EMBEDDING EXTRACTION SUMMARY

Train set:
  Text shape: (11674, 768)
  Image shape: (11674, 768)
  Labels: {0: np.int64(2756), 1: np.int64(8918)}

Val set:
  Text shape: (2502, 768)
  Image shape: (2502, 768)
  Labels: {0: np.int64(591), 1: np.int64(1911)}

Test set:
  Text shape: (2502, 768)
  Image shape: (2502, 768)
  Labels: {0: np.int64(591), 1: np.int64(1911)}

✓ Pipeline complete! Summary saved to c:\Users\Vivobook\Documents\mm_detect\data\embeddings\embedding_summary.json
